# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahmoud-Beram/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one content item's aggregate performance (`content_id`).

**Time window:** The month of March 2026 (Mid-panel month used to avoid outcome leakage from the final month).


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

* **Context:** `content_id`, `client_id` (IDs used for joining/grouping, never features).
* **Features:** `word_count`, `search_volume` (static traits), plus `total_impressions` (knowable before decision).
* **Label / Proxy:** Opportunity Score (Search Volume vs Clicks gap).
* **Excluded:** `trend_pct` and `trend_direction` (Leakage trap), and any future data after March 2026.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import duckdb
from google.colab import userdata

# 1. Connect to DuckDB
con = duckdb.connect()

# 2. Use the local file path that we already downloaded
# (Colab saves it in the huggingface cache)
import glob
local_file = glob.glob("/root/.cache/huggingface/hub/**/data_0.parquet", recursive=True)[0]

# ---------------------------------------------------------
# Query 1: Grain Check
# ---------------------------------------------------------
print("--- 1. Grain Check ---")
grain_df = con.execute(f"""
    SELECT content_hash_id, COUNT(*) as c
    FROM (
        SELECT content_hash_id
        FROM '{local_file}'
        GROUP BY content_hash_id
    )
    GROUP BY content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print("Duplicates found:" if not grain_df.empty else "Grain holds! No duplicates.")
print(grain_df)

# ---------------------------------------------------------
# Query 2: Row count and date span
# ---------------------------------------------------------
print("\n--- 2. Row count and date span ---")
span_df = con.execute(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) as total_articles,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM '{local_file}'
""").df()
print(span_df)

# ---------------------------------------------------------
# Query 3: Availability Check
# ---------------------------------------------------------
print("\n--- 3. Availability Check ---")
avail_df = con.execute(f"""
    SELECT
        COUNT(*) as total_raw_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as valid_ga4_rows
    FROM '{local_file}'
""").df()
print(avail_df)


--- 1. Grain Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain holds! No duplicates.
Empty DataFrame
Columns: [content_hash_id, c]
Index: []

--- 2. Row count and date span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_articles start_date   end_date
0          331437 2026-03-01 2026-03-31

--- 3. Availability Check ---
   total_raw_rows  valid_ga4_rows
0         9841378          413966


In [4]:
import duckdb
print(duckdb.execute("DESCRIBE SELECT * FROM '/root/.cache/huggingface/hub/**/data_0.parquet'").df()['column_name'].tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [5]:
# ---------------------------------------------------------
# Part 4: Five Features + The Leakage Trap
# ---------------------------------------------------------
print("\n--- 4. Feature Frame & Leakage Trap ---")

# 1. Build 5 honest features + 1 trap feature
features_df = con.execute(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(scroll_events) as total_scrolls,
        SUM(sessions_ai) as total_ai_traffic,

        -- TRAP: A feature derived directly from the label (total_clicks)
        CAST(SUM(gsc_clicks) > 500 AS INT) as TRAP_is_viral_leakage

    FROM '{local_file}'
    GROUP BY content_hash_id
""").df()

print("Feature frame built! Calculating Correlation with total_clicks...")
# 2. Demonstrate leakage (Trap will have near 1.0 correlation)
correlations = features_df.corr(numeric_only=True)['total_clicks'].sort_values(ascending=False)
print(correlations)

print("\n---------------------------------------------------------")
# 3. Drop the trap to keep honest features only
features_df = features_df.drop(columns=['TRAP_is_viral_leakage'])
print("The Trap has been deleted! Honest features remaining:")
display(features_df.head())



--- 4. Feature Frame & Leakage Trap ---
Feature frame built! Calculating Correlation with total_clicks...
total_clicks             1.000000
total_impressions        0.711723
TRAP_is_viral_leakage    0.567295
total_scrolls            0.469576
total_ai_traffic         0.113875
avg_position            -0.077978
Name: total_clicks, dtype: float64

---------------------------------------------------------
The Trap has been deleted! Honest features remaining:


,content_hash_id,total_impressions,total_clicks,avg_position,total_scrolls,total_ai_traffic
0,content_39d7361b4945d504,77.0,0.0,4.074107,NaN,NaN
1,content_cec711b02f3bbde6,602.0,4.0,4.428747,NaN,NaN
2,content_275b6f7f733016d4,810.0,1.0,4.866123,NaN,NaN
3,content_ceaec531566ffcfc,82.0,0.0,8.978086,NaN,NaN
4,content_755d951187fcd70a,1858.0,6.0,1.854929,NaN,NaN


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



**Unbalanced history**: We cannot fairly compare the total historical performance of two clients, because one might have joined 2 years ago while the other joined last month. The history depth is different per client.

**GSC-only early rows**: For periods before a client connected their GA4 (where ga4_data_available is False/NULL), the data cannot tell us anything about user engagement or time spent on the page. We only know they appeared on Google, but we are blind to what happened inside the site.

**Window overlaps**: We cannot use the raw 90-day search query data to strictly predict the future if that 90-day window overlaps with our target prediction month. The data cannot isolate exactly which searches happened strictly before the decision moment without specific window alignment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.